# Orthomosaic — Part 3: Cloud-Optimised GeoTIFF (COG)

**Convert the colour-corrected orthomosaic to COG layout using GeoBrix `rst_cog_convert`.**

**GeoBrix `rst_cog_convert`** wraps GDAL's native `driver="COG"` path via rasterio,
producing a spec-valid COG (internal tiles, overview pyramid, IFDs ordered for HTTP
range reads) without an extra rio-cogeo re-encode pass.  The result opens directly in
QGIS, GDAL, and any COG-aware client.

> **Prerequisite.** `02_color_correction` must have written `orthomosaic_corrected.tif`.

> **Runtime.** Runs on **Serverless environment 5**.  The corrected orthomosaic is loaded
> as a single-partition Spark DataFrame for `rst_cog_convert`.  For very large mosaics
> (> 1 GB) consider a tiled COG pipeline instead.

---

**Last Update:** September 21, 2026

## Setup

In [ ]:
%run ./config_nb

## Step 1: Generate COG using GeoBrix `rst_cog_convert`

In [ ]:
import time as _t_cog

_t0 = _t_cog.perf_counter()
try:
    from pathlib import Path as _P
    import shutil as _sh
    _groups = discover_groups()
    if not _groups:
        raise RuntimeError(f"no group_* under {output_dir} \u2014 run 01/02 first")
    for grp in _groups:
        _gp = group_paths(grp)
        _cog_out_dir = _gp["cog_dir"]
        _P(_cog_out_dir).mkdir(parents=True, exist_ok=True)
        # Read the corrected GeoTIFF as driver-local bytes (Volume FUSE). Spark's
        # binaryFile resolves bare paths as dbfs:, so a direct byte read + rst_fromcontent
        # keeps the GeoBrix rst_cog_convert demo.
        _content = _P(_gp["corrected"]).read_bytes()
        (
            spark.createDataFrame([("orthomosaic_cog", _content)], ["source", "content"])
                 .select("source", rx.rst_fromcontent(F.col("content"), F.lit("GTiff")).alias("tile"))
                 .select("source", rx.rst_cog_convert("tile").alias("tile"))
                 .write.format("gtiff_gbx").mode("overwrite").save(_cog_out_dir)
        )
        # gtiff_gbx names outputs by hash; normalize to the stable per-group cog name.
        _written = sorted(_P(_cog_out_dir).glob("*.tif"))
        if not _written:
            raise RuntimeError(f"group {grp!r}: gtiff_gbx produced no .tif under {_cog_out_dir}")
        _src = next((w for w in _written if w.name == _P(_gp["cog"]).name), _written[0])
        if _src.resolve() != _P(_gp["cog"]).resolve():
            _sh.move(str(_src), _gp["cog"])
        print(f"  group {grp!r}: COG \u2192 {_gp['cog']}")
    print(f"COG(s) written in {_t_cog.perf_counter()-_t0:.1f}s ({len(_groups)} group(s))")
except Exception as e:
    print(f"[ERROR] COG step failed after {_t_cog.perf_counter()-_t0:.1f}s: {e}")
    raise

## Step 2: Visualise and validate

In [ ]:
vz.plot_cog(group_paths(discover_groups()[0])["cog"])

## Steps performed

1. **COG conversion** — `rst_cog_convert` (GeoBrix) applied GDAL's `driver="COG"` path with
   DEFLATE compression, 512-px internal tiles, and an AVERAGE overview pyramid.
2. **Preview** — `vz.plot_cog` (GeoBrix VizX) renders the Cloud-Optimized GeoTIFF over a basemap.

**Next:** Part 4 converts the COG to PMTiles for interactive map serving.